In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd

file_path = "/content/drive/MyDrive/Ds/financial_news_base.jsonl"

df = pd.read_json(file_path, lines=True)


In [4]:
df_bert = df[["description","sentiment_label"]]



In [5]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [7]:
from sklearn.model_selection import train_test_split
X = df_bert["description"]
y = df_bert["sentiment_label"].replace({"negative" : 0 , "neutral":1, "positive" : 2})

X_train, X_temp, y_train, y_temp = train_test_split(X,y, test_size = 0.3, random_state = 42, stratify= y)

X_val, X_test, y_val, y_test = train_test_split(X_temp,y_temp, test_size = 0.5, random_state = 42, stratify = y_temp)

print(X_train.shape, X_val.shape, X_test.shape)
y_train.head()


(49382,) (10582,) (10583,)


/tmp/ipykernel_7760/2534941598.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = df_bert["sentiment_label"].replace({"negative" : 0 , "neutral":1, "positive" : 2})


,sentiment_label
20075,2
122,0
24862,2
34433,0
9026,2


In [8]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

batch_texts = X_train.tolist()
train_tokenized = tokenizer(batch_texts,
                      padding="max_length",
                      truncation=True,
                      return_tensors="pt")

val_tokenized = tokenizer(
    X_val.tolist(),
    truncation=True,
    padding="max_length",
    return_tensors="pt"
)

test_tokenized = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding="max_length",
    return_tensors="pt"
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [9]:
y_train_tensor = torch.tensor(y_train.values)
y_val_tensor = torch.tensor(y_val.values)
y_test_tensor = torch.tensor(y_test.values)

In [10]:
class Dataset(torch.utils.data.Dataset):
  def __init__(self,tokens, labels):
    self.tokens = tokens
    self.labels = labels

  def __getitem__(self, idx):
      item = {key: val[idx] for key, val in self.tokens.items()}
      item["labels"] = self.labels[idx]
      return item

  def __len__(self):
      return len(self.labels)

In [11]:
train_dataset = Dataset(train_tokenized, y_train_tensor)
val_dataset = Dataset(val_tokenized, y_val_tensor)
test_dataset = Dataset(test_tokenized, y_test_tensor)


train_loader = DataLoader(train_dataset, batch_size = 16)
val_loader = DataLoader(val_dataset, batch_size = 16)
test_loader = DataLoader(test_dataset, batch_size = 16)

In [12]:
from transformers import BertConfig
config_finetune = BertConfig.from_pretrained('bert-base-uncased', num_labels=3, output_attentions=True)
model_finetune = BertForSequenceClassification.from_pretrained("bert-base-uncased", config = config_finetune)
for name, param in model_finetune.bert.named_parameters():
    layer_num = int(name.split('.')[2]) if 'layer.' in name else None

    if layer_num is not None and layer_num < 8:
        param.requires_grad = False
model_finetune.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [13]:
print(f"\n🔢 Total parameters: {sum(p.numel() for p in model_finetune.parameters()):,}")
print(f"🔢 Trainable parameters: {sum(p.numel() for p in model_finetune.parameters() if p.requires_grad):,}")


🔢 Total parameters: 109,484,547
🔢 Trainable parameters: 52,781,571


In [14]:
from transformers import get_linear_schedule_with_warmup
optimizer = AdamW(model_finetune.parameters(), lr=2e-5)
epochs = 3
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

In [15]:
import time
from tqdm import tqdm


def train_epoch(model ,dataloader, optimizer, scheduler, device):
  model.train()
  total_loss = 0

  for batch in tqdm(dataloader, desc = "Training"):
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    optimizer.zero_grad()

        # Forward pass: compute model predictions and loss
        # BertForSequenceClassification automatically computes loss when labels are provided
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss

        # Backward pass: compute gradients
    loss.backward()

        # Update weights based on gradients
    optimizer.step()

    scheduler.step()

    total_loss += loss.item()

  return total_loss / len(dataloader)

In [16]:
finetune_losses = []
start_time = time.time()

for epoch in range(epochs):
    print(f"\n📍 Epoch {epoch + 1}/{epochs}")
    loss = train_epoch(model_finetune, train_loader, optimizer, scheduler, device)
    finetune_losses.append(loss)
    print(f"  Average loss: {loss:.4f}")

finetune_time = time.time() - start_time


📍 Epoch 1/20


Training:   0%|          | 0/1544 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 48.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 1.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 14.34 GiB is allocated by PyTorch, and 97.25 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)